# Data Exploration — MVTec AD

This notebook explores the structure, statistics, and visual characteristics of the
[MVTec Anomaly Detection dataset](https://www.mvtec.com/company/research/datasets/mvtec-ad)
before training any model.

**Goals:**
1. Understand dataset scale — how many samples per category, per split
2. Visualise the class imbalance between normal and anomalous test samples
3. Inspect image content: normal images, defect images, and ground-truth masks
4. Confirm image size and channel distribution after transforms

---

In [ ]:
import sys
from pathlib import Path

# Make sure the project root is on the path
ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

from src.data.dataset import CATEGORIES, MVTecADDataset
from src.data.transforms import get_transforms

DATA_ROOT = ROOT / "data"
print(f"Data root : {DATA_ROOT}")
print(f"Categories: {len(CATEGORIES)}")

## 1. Dataset Size Overview

Count training and test images for every category.

In [ ]:
stats = []

for cat in CATEGORIES:
    cat_dir = DATA_ROOT / cat
    if not cat_dir.exists():
        print(f"  [skip] {cat} — not downloaded yet")
        continue

    n_train  = len(list((cat_dir / "train" / "good").glob("*.png")))
    test_dir = cat_dir / "test"
    n_normal = len(list((test_dir / "good").glob("*.png"))) if (test_dir / "good").exists() else 0
    n_anom   = sum(
        len(list(d.glob("*.png")))
        for d in test_dir.iterdir()
        if d.is_dir() and d.name != "good"
    )
    defect_types = [d.name for d in test_dir.iterdir() if d.is_dir() and d.name != "good"]

    stats.append({
        "category":     cat,
        "train":        n_train,
        "test_normal":  n_normal,
        "test_anom":    n_anom,
        "defect_types": len(defect_types),
    })
    print(f"  {cat:15s}  train={n_train:4d}  test_normal={n_normal:3d}  test_anom={n_anom:3d}  defects={len(defect_types)}")

In [ ]:
if stats:
    cats        = [s["category"]    for s in stats]
    n_train     = [s["train"]       for s in stats]
    n_normal    = [s["test_normal"] for s in stats]
    n_anom      = [s["test_anom"]   for s in stats]

    x = np.arange(len(cats))
    w = 0.28

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - w, n_train,  width=w, label="Train (normal)",  color="steelblue")
    ax.bar(x,     n_normal, width=w, label="Test (normal)",   color="mediumseagreen")
    ax.bar(x + w, n_anom,   width=w, label="Test (anomalous)",color="tomato")

    ax.set_xticks(x)
    ax.set_xticklabels(cats, rotation=40, ha="right", fontsize=9)
    ax.set_ylabel("Image count")
    ax.set_title("MVTec AD — Sample Counts per Category")
    ax.legend()
    plt.tight_layout()
    plt.show()

    total_train = sum(n_train)
    total_test  = sum(n_normal) + sum(n_anom)
    print(f"\nTotal training images : {total_train:,}")
    print(f"Total test images     : {total_test:,}")
    print(f"  - Normal            : {sum(n_normal):,}")
    print(f"  - Anomalous         : {sum(n_anom):,}")
    print(f"\nObservation: test sets are heavily skewed toward anomalous samples.")
    print(f"This is expected — real production lines see mostly normal parts.")

## 2. Anomaly Type Distribution

How many distinct defect types does each category have?

In [ ]:
if stats:
    cats_s  = [s["category"]    for s in stats]
    n_types = [s["defect_types"] for s in stats]

    fig, ax = plt.subplots(figsize=(12, 4))
    bars = ax.barh(cats_s, n_types, color="mediumpurple")
    ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_xlabel("Number of defect types")
    ax.set_title("Defect Types per Category")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    print(f"Average defect types per category: {np.mean(n_types):.1f}")
    print(f"Max: {max(n_types)} ({cats_s[np.argmax(n_types)]})")

## 3. Visual Inspection — Normal vs Anomalous

Let's look at some example images alongside their ground-truth masks.

In [ ]:
def show_category_samples(category: str, n_per_defect: int = 2):
    """Display normal and anomalous samples for a category."""
    cat_dir  = DATA_ROOT / category
    test_dir = cat_dir / "test"
    gt_dir   = cat_dir / "ground_truth"

    if not test_dir.exists():
        print(f"Dataset not downloaded — skipping {category}")
        return

    defect_types = sorted([d.name for d in test_dir.iterdir() if d.is_dir() and d.name != "good"])
    normal_imgs  = sorted((test_dir / "good").glob("*.png"))[:2]

    # Gather defect examples (1 per type, up to 4 types)
    defect_examples = []
    for dtype in defect_types[:4]:
        imgs = sorted((test_dir / dtype).glob("*.png"))
        if imgs:
            defect_examples.append((dtype, imgs[0]))

    n_cols = 2 + len(defect_examples)   # normal cols + defect cols
    n_rows = 2                           # image row + mask row

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 7))
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    fig.suptitle(f"Category: {category}", fontsize=14, fontweight="bold")

    # --- Normal samples (columns 0, 1) ---
    for col, img_path in enumerate(normal_imgs):
        img = Image.open(img_path).convert("RGB").resize((224, 224))
        axes[0, col].imshow(img)
        axes[0, col].set_title(f"Normal ({img_path.stem})", fontsize=8)
        axes[0, col].axis("off")
        # Empty mask row
        axes[1, col].imshow(np.zeros((224, 224)), cmap="gray", vmin=0, vmax=1)
        axes[1, col].set_title("No defect mask", fontsize=8)
        axes[1, col].axis("off")

    # --- Defect samples ---
    for col_offset, (dtype, img_path) in enumerate(defect_examples):
        col = 2 + col_offset
        img = Image.open(img_path).convert("RGB").resize((224, 224))
        axes[0, col].imshow(img)
        axes[0, col].set_title(f"{dtype}\n({img_path.stem})", fontsize=8)
        axes[0, col].axis("off")

        # Ground truth mask
        mask_path = gt_dir / dtype / (img_path.stem + "_mask.png")
        if mask_path.exists():
            mask = Image.open(mask_path).convert("L").resize((224, 224))
            axes[1, col].imshow(np.array(mask), cmap="Reds", vmin=0, vmax=255)
            axes[1, col].set_title("GT mask", fontsize=8)
        else:
            axes[1, col].text(0.5, 0.5, "mask not found", ha="center", va="center", transform=axes[1, col].transAxes)
        axes[1, col].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Inspect a few representative categories
for cat in ["bottle", "cable", "leather", "tile"]:
    show_category_samples(cat)

## 4. Tensor Shape After Transforms

Verify that the data pipeline produces tensors of the expected shape and value range.

In [ ]:
from src.data.transforms import get_transforms

transforms = get_transforms(image_size=224, center_crop=224)

try:
    ds = MVTecADDataset(
        root=str(DATA_ROOT),
        category="bottle",
        split="train",
        image_size=224,
        center_crop=224,
    )
    sample = ds[0]

    print("Sample keys      :", list(sample.keys()))
    print("Image shape      :", sample["image"].shape)    # (3, 224, 224)
    print("Image dtype      :", sample["image"].dtype)
    print("Image min/max    :", sample["image"].min().item(), "/", sample["image"].max().item())
    print("Mask shape       :", sample["mask"].shape)     # (1, 224, 224)
    print("Label            :", sample["label"])
    print("Image path       :", sample["image_path"])
except FileNotFoundError:
    print("Dataset not found. Run: python scripts/download_dataset.py --root data")

## 5. Key Takeaways

From exploring the raw data:

- **No augmentation on training data.** PatchCore trains on unaugmented images because
  the memory bank should capture the exact feature distribution of good parts — flips
  or crops could introduce artefacts that don't exist on the production line.

- **Strong class imbalance.** Anomalous test images typically outnumber normal test images
  ~2:1. AUROC handles this well, but F1 requires a threshold — which is why we compute
  the *optimal* F1 threshold post-hoc in `evaluate.py`.

- **Pixel-level masks are sparse.** Defects are often small (screws, capsules) relative
  to the image. The Gaussian smoothing in `predict()` helps propagate the anomaly signal
  from a few patches to their neighbourhood.

- **Category diversity is intentional.** MVTec spans textures (carpet, leather) and objects
  (bottle, cable). PatchCore's strength is generalising across both without category-specific
  tuning — which is exactly what this implementation tests.